In [2]:
# 1. Connect to a MySQL Database and Query Data

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import getpass
import matplotlib.pyplot as plt

bd = "sakila"             
user = "root"                 
host = "127.0.0.1"             
port = 3306


pw_raw = getpass.getpass("MySQL password: ")
pw = quote_plus(pw_raw)

url = f"mysql+pymysql://{user}:{pw}@{host}:{port}/{bd}?charset=utf8mb4"


engine = create_engine(url, pool_pre_ping=True)

with engine.begin() as conn:
    print(conn.exec_driver_sql("SELECT 1").scalar())

1


In [7]:
def rentals_month(engine, month: int, year:int):
    """
    Retrieves rental data for a given month and year (passed as parameters) from the Sakila database as a Pandas DataFrame.

    Execute a SQL query to retrieve the rental data for the specified month and year from the rental table in the Sakila database, 
    and return it as a pandas DataFrame.

    """
    query = f"""
        SELECT
            *
        FROM rental
        WHERE YEAR(rental_date) = {year}
          AND MONTH(rental_date) = {month}
        ORDER BY rental_date;
    """ 
    with engine.connect() as conn:
        df = pd.read_sql(query, conn)
    return df



In [8]:
# Trial to see if the function works properly:

df = rentals_month(engine, 6, 2005)

print(df.head())



   rental_id         rental_date  inventory_id  customer_id  \
0       1158 2005-06-14 22:53:33          1632          416   
1       1159 2005-06-14 22:55:13          4395          516   
2       1160 2005-06-14 23:00:34          2795          239   
3       1161 2005-06-14 23:07:08          1690          285   
4       1162 2005-06-14 23:09:38           987          310   

          return_date  staff_id         last_update  
0 2005-06-18 21:37:33         2 2006-02-15 21:30:53  
1 2005-06-17 02:11:13         1 2006-02-15 21:30:53  
2 2005-06-18 01:58:34         2 2006-02-15 21:30:53  
3 2005-06-21 17:12:08         1 2006-02-15 21:30:53  
4 2005-06-23 22:00:38         1 2006-02-15 21:30:53  


In [10]:
import pandas as pd

def rental_count_month(df: pd.DataFrame, month: int, year: int) -> pd.DataFrame:
    """
    Returns a DataFrame showing the number of rentals per customer_id
    for the specified month and year.
    """

    # Create a dynamic column name
    col_name = f"rentals_{month:02d}_{year}"

    # Group by customer and count how many rentals each made
    result = (
        df.groupby("customer_id")
          .size()  # counts the number of rows per customer_id
          .reset_index(name=col_name)
    )

    return result


In [11]:

# Now get the rental counts per customer for that month:
df_counts = rental_count_month(df, 5, 2005)

print(df_counts.head())


   customer_id  rentals_05_2005
0            1                7
1            2                1
2            3                4
3            4                6
4            5                5


In [ ]:
import pandas as pd

def compare_rentals(df1: pd.DataFrame, df2: pd.DataFrame):
    """
    Combines two DataFrames containing rental counts by customer for
    different months/years and calculates the difference.

    """

    # Merge the two DataFrames on customer_id (keeps all customers)
    combined = pd.merge(df1, df2, on="customer_id", how="outer").fillna(0)

    # Get the names of the rental columns dynamically
    col1 = [c for c in df1.columns if c != "customer_id"][0]
    col2 = [c for c in df2.columns if c != "customer_id"][0]

    # Create the difference column
    combined["difference"] = combined[col2] - combined[col1]

    return combined
